AI Agents from First Principles

**AIYatra Meetup · SelfEvaluate Agent: Build a safe Agent with LangChain, LangGraph & LangSmith**
📍 London Stock Exchange Group, Hyderabad · 🗓️ Saturday 31 Oct 2026 · ⏰ 9:00 AM to 12:30 PM IST

> *"What happens when an AI agent can read and modify its own source code?"*

Today we answer that question by building a **self-editing coding agent** that inspects files, applies changes, runs code and checks its own work, **safely**.
Before touching any framework, this notebook builds the core ideas **from scratch in plain Python**. Then you'll know exactly which problem each framework solves.

##  The three tools at a glance

| | **LangChain** | **LangGraph** | **LangSmith** |
|---|---|---|---|
| **Role** | The *components*: models, prompts, tools, retrievers, agents | The *runtime and control flow*: state, loops, branching, pausing, persistence | The *eyes and the exam*: tracing, debugging, datasets, evaluation |
| **Analogy** | 🧱 LEGO bricks | 🗺️ The blueprint and the foreman | 🔬 The microscope and the report card |
| **Open source?** | ✅ MIT | ✅ MIT | SDK ✅ MIT, platform is SaaS or self-hosted |
| **Use without the others?** | Yes | Yes (works with any Python code) | Yes (traces *any* code, not just LangChain) |

They are **independent but designed to click together**. LangChain's `create_agent` actually *compiles to a LangGraph graph*, and both emit traces that LangSmith understands with zero extra code.



## 1 · Environment setup

### 1.1 Install packages
Run once. Python **3.10+** is required, and 3.12 is recommended.

In [ ]:
%pip install -qU langchain langchain-core langgraph langsmith langchain-text-splitters \
    langchain-openai langchain-anthropic langchain-ollama python-dotenv httpx "pandas>=2,<3"

### 1.2 Choose a model, any model 🔌

Everything in this workshop is **model-agnostic**. Create a file called `.env` next to the notebooks with **one** of these:

```bash
# Option A: OpenAI
OPENAI_API_KEY=sk-...
# Option B: Anthropic
ANTHROPIC_API_KEY=sk-ant-...
# Option C: Google Gemini (also: pip install langchain-google-genai)
GOOGLE_API_KEY=...
# Option D: Groq (also: pip install langchain-groq)
GROQ_API_KEY=...
# Option E: nothing at all → free local model via Ollama (https://ollama.com)
#   ollama pull llama3.2:3b     (small and fast; llama3.1:8b or qwen2.5:14b are smarter)

# Optional overrides
LLM_PROVIDER=openai          # force a provider
LLM_MODEL=gpt-4.1-mini       # force a model name

# LangSmith (free account at https://smith.langchain.com), used heavily in notebooks 02 and 04
LANGSMITH_API_KEY=lsv2_...
```

The cell below is **the same in every notebook**. It reads `.env`, picks a provider, and exposes `get_llm()`.

In [ ]:
# ── Shared workshop setup (identical in every notebook) ───────────────────────
# Model-agnostic: picks a provider from LLM_PROVIDER or whichever API key it finds,
# and falls back to a free local Ollama model. Override with LLM_PROVIDER / LLM_MODEL.
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from dotenv import load_dotenv
    load_dotenv()  # reads a .env file next to the notebook, if present
except ImportError:
    pass

DEFAULT_MODELS = {
    "openai": "gpt-4.1-mini",
    "anthropic": "claude-sonnet-5-5",
    "google_genai": "gemini-2.5-flash",
    "groq": "llama-3.3-70b-versatile",
    "ollama": "llama3.2:3b",
}


def pick_provider() -> str:
    if os.getenv("LLM_PROVIDER"):
        return os.environ["LLM_PROVIDER"]
    for env_key, provider in [
        ("OPENAI_API_KEY", "openai"),
        ("ANTHROPIC_API_KEY", "anthropic"),
        ("GOOGLE_API_KEY", "google_genai"),
        ("GROQ_API_KEY", "groq"),
    ]:
        if os.getenv(env_key):
            return provider
    return "ollama"


PROVIDER = pick_provider()
MODEL = os.getenv("LLM_MODEL", DEFAULT_MODELS[PROVIDER])


def get_llm(temperature: float = 0, **kwargs):
    """Return a LangChain chat model for whichever provider is configured."""
    from langchain.chat_models import init_chat_model

    if PROVIDER == "ollama":
        kwargs.setdefault("num_predict", 2048)  # stop runaway generations on small local models
    return init_chat_model(MODEL, model_provider=PROVIDER, temperature=temperature, **kwargs)


# LangSmith tracing turns on automatically when a key is present.
LANGSMITH_ON = bool(os.getenv("LANGSMITH_API_KEY"))
if LANGSMITH_ON:
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "aiyatra-selfevaluate-workshop")
else:
    os.environ["LANGSMITH_TRACING"] = "false"

print(f"LLM provider : {PROVIDER}")
print(f"LLM model    : {MODEL}")
print(f"LangSmith    : {'ON → project ' + os.environ['LANGSMITH_PROJECT'] if LANGSMITH_ON else 'off (set LANGSMITH_API_KEY to enable)'}")

---
#  Part 1: What *is* an LLM API?

Strip away the hype and a Large Language Model is a **function**:


Under the hood it predicts one **token** (a word-piece of about 4 characters) at a time, appending each to the input, until it emits a stop token. The API wraps that loop in a single HTTP request.

Three facts drive everything that follows:
1. **It's just HTTP + JSON.** No magic SDK is required.
2. **It's stateless.** The model remembers *nothing* between calls. "Memory" means *you* resend the history.
3. **Every provider's JSON is slightly different.** Hold that thought. It's why LangChain exists.

### 1.1 A raw HTTP call, no SDK
Most providers (OpenAI, Groq, Gemini, Ollama…) speak the **OpenAI-compatible** `/chat/completions` format. Anthropic uses its own `/v1/messages` format. Below we hand-write an adapter for both.

In [ ]:
import json

import httpx

# OpenAI-compatible endpoints: (base_url, env var holding the key)
OPENAI_COMPATIBLE = {
    "openai": ("https://api.openai.com/v1", "OPENAI_API_KEY"),
    "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY"),
    "google_genai": ("https://generativelanguage.googleapis.com/v1beta/openai", "GOOGLE_API_KEY"),
    "ollama": (os.getenv("OLLAMA_BASE_URL", "http://localhost:11434") + "/v1", None),
}


def _to_anthropic(messages):
    """Translate OpenAI-style messages into Anthropic's format (a taste of the 'glue code' tax)."""
    out = []
    for m in messages:
        if m["role"] == "system":
            continue
        if m["role"] == "tool":
            block = {"type": "tool_result", "tool_use_id": m["tool_call_id"], "content": m["content"]}
            if out and out[-1]["role"] == "user" and isinstance(out[-1]["content"], list):
                out[-1]["content"].append(block)  # consecutive tool results share one user turn
            else:
                out.append({"role": "user", "content": [block]})
        elif m["role"] == "assistant" and m.get("tool_calls"):
            blocks = [{"type": "text", "text": m["content"]}] if m.get("content") else []
            blocks += [
                {"type": "tool_use", "id": c["id"], "name": c["function"]["name"],
                 "input": json.loads(c["function"]["arguments"])}
                for c in m["tool_calls"]
            ]
            out.append({"role": "assistant", "content": blocks})
        else:
            out.append({"role": m["role"], "content": m["content"]})
    return out


def raw_chat(messages, tools=None, temperature=0.0):
    """Call the configured provider over plain HTTP and normalise the reply to
    {"content": str, "tool_calls": [{"id", "name", "args"}], "usage": dict}."""
    if PROVIDER == "anthropic":
        body = {"model": MODEL, "max_tokens": 1024, "temperature": temperature,
                "messages": _to_anthropic(messages)}
        system = "\n".join(m["content"] for m in messages if m["role"] == "system")
        if system:
            body["system"] = system
        if tools:
            body["tools"] = [{"name": t["function"]["name"],
                              "description": t["function"]["description"],
                              "input_schema": t["function"]["parameters"]} for t in tools]
        r = httpx.post("https://api.anthropic.com/v1/messages", json=body, timeout=120,
                       headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"],
                                "anthropic-version": "2023-06-01"})
        r.raise_for_status()
        data = r.json()
        return {
            "content": "".join(b["text"] for b in data["content"] if b["type"] == "text"),
            "tool_calls": [{"id": b["id"], "name": b["name"], "args": b["input"]}
                           for b in data["content"] if b["type"] == "tool_use"],
            "usage": data.get("usage", {}),
        }

    base_url, key_env = OPENAI_COMPATIBLE[PROVIDER]
    headers = {"Authorization": f"Bearer {os.environ[key_env]}"} if key_env else {}
    body = {"model": MODEL, "messages": messages, "temperature": temperature}
    if tools:
        body["tools"] = tools
    r = httpx.post(f"{base_url}/chat/completions", json=body, headers=headers, timeout=120)
    r.raise_for_status()
    msg = r.json()["choices"][0]["message"]
    return {
        "content": msg.get("content") or "",
        "tool_calls": [{"id": c["id"], "name": c["function"]["name"],
                        "args": json.loads(c["function"]["arguments"] or "{}")}
                       for c in (msg.get("tool_calls") or [])],
        "usage": r.json().get("usage", {}),
    }


reply = raw_chat([
    {"role": "system", "content": "You are a concise teaching assistant."},
    {"role": "user", "content": "In one sentence, what is an AI agent?"},
])
print(reply["content"])
print("\ntoken usage:", reply["usage"])

### 1.2 Roles: the message types
| role | who | purpose |
|------|-----|---------|
| `system` | developer | Rules. Applicable to whole chat. "You are a concise assistant." |
| `user` | human | Question or request. |
| `assistant` | model | Model reply. Sometimes plain text, sometimes  **tool calls** |
| `tool` | your code | The *result* of running a tool, fed back to the model |

### 1.3 Proving statelessness
Watch the model "forget" unless we resend the history ourselves.

In [ ]:
print("❌ Without history:")
print(raw_chat([{"role": "user", "content": "What is my name?"}])["content"][:200])

history = [
    {"role": "user", "content": "Hi! My name is Asha and I love Rust."},
    {"role": "assistant", "content": "Nice to meet you, Asha! Rust is a great language."},
    {"role": "user", "content": "What is my name and favourite language?"},
]
print("\n✅ With history resent by *us*:")
print(raw_chat(history)["content"][:200])

>  **Takeaway:** "Conversation memory" is just *you* storing and resending a list. Later, LangGraph **checkpointers** do this for you per `thread_id`, durably.

### 1.4 Temperature from first principles
The model outputs a score (**logit**) for every possible next token. **Softmax** turns those into probabilities, and **temperature** reshapes them before sampling:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

Low `T` → nearly always the top token (deterministic, good for code). High `T` → flatter distribution (creative, risky).

In [ ]:
import math

logits = {"fix": 4.0, "patch": 3.2, "rewrite": 2.0, "delete": 0.5}


def softmax(scores, T):
    exps = {k: math.exp(v / T) for k, v in scores.items()}
    total = sum(exps.values())
    return {k: v / total for k, v in exps.items()}


for T in (0.2, 1.0, 2.0):
    probs = softmax(logits, T)
    bars = "  ".join(f"{k}:{'█' * round(p * 30):<30}{p:5.1%}" for k, p in probs.items())
    print(f"T={T:<4} {bars}")

> For a **coding agent** we use `temperature=0`. We want the most likely fix, not a creative one. That's why `get_llm()` defaults to 0.

---
#  Part 2: Tool calling from scratch

An LLM can only produce text. So how does an "agent" run code or read files?

**Tool calling** is a contract:
1. You describe tools as **JSON Schema** (name, description, parameters).
2. The model, instead of answering, replies with *"please call `tool_x` with `{args}`"*.
3. **Your code** executes the tool. The model never runs anything itself.
4. You send the result back as a `tool` message, and the model continues.

>  **Safety insight #1:** Because *your* code executes every tool, *your* code is where guardrails belong. A prompt saying "please don't delete files" is a suggestion. A path check in the tool is a law.

In [ ]:
# 1. Plain Python functions: the "tools"
def add(a: float, b: float) -> float:
    return a + b


def multiply(a: float, b: float) -> float:
    return a * b


def word_count(text: str) -> int:
    return len(text.split())


PY_TOOLS = {"add": add, "multiply": multiply, "word_count": word_count}


# 2. JSON Schemas, written by hand. LangChain's @tool will generate these for us later.
def schema(name, description, properties, required):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": required}}}


num = {"type": "number"}
TOOL_SCHEMAS = [
    schema("add", "Add two numbers.", {"a": num, "b": num}, ["a", "b"]),
    schema("multiply", "Multiply two numbers.", {"a": num, "b": num}, ["a", "b"]),
    schema("word_count", "Count words in a text.", {"text": {"type": "string"}}, ["text"]),
]
print(json.dumps(TOOL_SCHEMAS[0], indent=2))

In [ ]:
# 3. Ask a question that needs a tool, and look at what comes back
reply = raw_chat([{"role": "user", "content": "What is 1234 multiplied by 5678? Use a tool."}], tools=TOOL_SCHEMAS)
print("content   :", repr(reply["content"][:120]))
print("tool_calls:", reply["tool_calls"])

The model didn't answer. It **asked us to call a function**. Now we close the loop.

---
#  Part 3: The agent loop, hand-rolled

An **agent** is an LLM in a `while` loop:

```
messages = [system, user]
loop:
    reply = LLM(messages, tools)
    if reply has no tool calls → return reply           # done
    for each tool call → run it, append result          # act
    (repeat, with a step budget so it can't run forever)
```

That's the famous **ReAct** pattern: *Reason → Act → Observe → repeat*. Here it is in about 30 lines.

>  **Models are sloppy with types.** Small models often send `{"a": "12"}` (a *string*) for a `number` parameter. So we validate and coerce arguments against the schema before running a tool. LangChain's `@tool` does this automatically with Pydantic.

In [ ]:
SCHEMA_BY_NAME = {t["function"]["name"]: t["function"]["parameters"] for t in TOOL_SCHEMAS}


def coerce_args(tool_name: str, args: dict) -> dict:
    """Validate/coerce args against the JSON schema: a mini version of what Pydantic does."""
    props = SCHEMA_BY_NAME[tool_name]["properties"]
    missing = set(SCHEMA_BY_NAME[tool_name]["required"]) - set(args)
    if missing:
        raise ValueError(f"missing arguments: {sorted(missing)}")
    return {k: float(v) if props.get(k, {}).get("type") == "number" else v for k, v in args.items()}


def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": "You are a precise assistant. Use tools for any arithmetic. "
                                      "When you have the final answer, reply in plain text."},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        reply = raw_chat(messages, tools=TOOL_SCHEMAS)
        if not reply["tool_calls"]:
            if verbose:
                print(f"✅ step {step}: final answer")
            return reply["content"]

        # Record the assistant's request in OpenAI format (our neutral internal format)
        messages.append({"role": "assistant", "content": reply["content"], "tool_calls": [
            {"id": c["id"], "type": "function",
             "function": {"name": c["name"], "arguments": json.dumps(c["args"])}}
            for c in reply["tool_calls"]]})

        for call in reply["tool_calls"]:
            fn = PY_TOOLS.get(call["name"])
            try:
                result = fn(**coerce_args(call["name"], call["args"])) if fn else f"ERROR: unknown tool {call['name']}"
            except Exception as e:  # feed errors back: the model can often self-correct
                result = f"ERROR: {e}"
            if verbose:
                print(f"🔧 step {step}: {call['name']}({call['args']}) → {result}")
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": str(result)})
    return " Stopped: step budget exhausted."


print(run_agent("What is (12 + 30) multiplied by 3? Then add 100 to that."))

 **You just built an agent from first principles.** No frameworks.

##  So why do we need frameworks?

Try to grow this toy into the **SelfEvaluate coding agent** and the cracks show:

| Pain in our hand-rolled agent | Who fixes it | How |
|---|---|---|
| A hand-written adapter per provider (look at `_to_anthropic`!) | **LangChain** | `init_chat_model("provider:model")`, one interface for 100+ models |
| Hand-written JSON schemas for tools | **LangChain** | `@tool` reads type hints and docstrings |
| Hand-written argument validation (`coerce_args`) | **LangChain** | Pydantic validates every tool call |
| Parsing JSON out of model text | **LangChain** | `.with_structured_output(PydanticModel)` |
| Guardrails scattered inside the loop | **LangChain** | Agent **middleware** (`@wrap_tool_call`, call limits, HITL) |
| A loop that can only go *forward*. No "verify → fix → verify again", no branching | **LangGraph** | Explicit **graph**: nodes, conditional edges, cycles |
| History lost when the process dies, and no undo | **LangGraph** | **Checkpointers**: memory, resume, time travel |
| "Pause and ask a human before editing 5 files" | **LangGraph** | `interrupt()` + `Command(resume=...)` |
| `print()` debugging: "why did it call that tool?!" | **LangSmith** | **Traces**: every LLM call, tool call, token and latency in a tree |
| "Is my new prompt better or worse?" Vibes only | **LangSmith** | **Datasets + evaluators + experiments** |

##  The SelfEvaluate agent's safety model (preview)

When an agent can **edit its own code**, the guardrails must live *outside* the model's control:

1. **Path jail**: every path is resolved and must stay inside the workspace (`../../etc/passwd` ❌)
2. **Protected files**: the guardrail code and the tests are read-only to the agent. It can't "fix" tests to pass, or remove its own safety checks
3. **Syntax gate**: `ast.parse()` runs before any `.py` file is written
4. **Automatic backups**: a copy before every write, plus a workspace snapshot and rollback
5. **Run, verify, fix**: execute tests in a sandboxed subprocess with a timeout, then loop back on failure
6. **Budgets**: max iterations and max tool calls, so it can't loop forever
7. **Human gate**: pause for approval on risky multi-file changes
8. **Observability + evals**: trace everything and score it on a dataset of tasks

